# Asymmetric Multilingual Acquisition: Typology-Aware Adaptive Mixing for BabyLMs

- Get HF_TOKEN with `write-access`

In [ ]:
# Cell 1 — Parameters
# -------------------
# What/where to evaluate. Edit before running.

HF_NAMESPACE = "amosluna"

# Five final models (main branch = final checkpoint, ~655M tokens).
# Zero-shot over these is the FIRST thing to run (~20-30 min each).
FINAL_MODELS = [
    f"{HF_NAMESPACE}/babylm-2026-taam-seed42",
    f"{HF_NAMESPACE}/babylm-2026-taam-seed1337",
    f"{HF_NAMESPACE}/babylm-2026-taam_v2-seed42",
    f"{HF_NAMESPACE}/babylm-2026-taam_v2-seed1337",
    f"{HF_NAMESPACE}/babylm-2026-b0-seed42",
]

# Key models to evaluate per intermediate checkpoint (H4 = acquisition order).
# Only 2 — each takes ~6-8h because of the 24 chck_NM revisions.
H4_MODELS = [
    f"{HF_NAMESPACE}/babylm-2026-taam_v2-seed42",
    f"{HF_NAMESPACE}/babylm-2026-b0-seed42",
]

LANGS = "eng nld zho"

# Flags to run selectively (Colab sometimes disconnects; use these).
RUN_ZEROSHOT_FINAL        = True   # Step 1 — FAST, priority #1 / False for H4
RUN_ZEROSHOT_INTERMEDIATE = False  # Step 3 — SLOW (~16h total), run later / True for H4
RUN_FINETUNE              = False  # Optional — VERY SLOW (~3h per model/lang) / True for Fine-tunning
RUN_COLLATE               = True   # Build *_submission.json at the end
RUN_SUMMARY_TABLE         = True   # Comparative table at the end

# Persistence on Drive (recommended so we don't lose results on disconnect).
USE_DRIVE   = True
DRIVE_ROOT  = "/content/drive/MyDrive/Researchs/BabyLM"
EVAL_DIR    = "/content/babylm-eval"
RESULTS_DIR = f"{DRIVE_ROOT}/eval_results"  # symlink ← babylm-eval/results/

print("Plan:")
print(f"  final zero-shot       : {RUN_ZEROSHOT_FINAL}  ({len(FINAL_MODELS)} models)")
print(f"  intermediate eval (H4): {RUN_ZEROSHOT_INTERMEDIATE}  ({len(H4_MODELS)} models)")
print(f"  finetune              : {RUN_FINETUNE}")
print(f"  collate submissions   : {RUN_COLLATE}")
print(f"  summary table         : {RUN_SUMMARY_TABLE}")
print(f"  results persisted in  : {RESULTS_DIR if USE_DRIVE else '/content (ephemeral!)'}")

In [ ]:
# Cell 2 — Mount Drive + clone babylm-eval + install lm-eval-harness
# ----------------------------------------------------------------
# Idempotent: safe to re-run this cell at any time.

import os, sys, shutil, subprocess, shlex
from pathlib import Path

def _brief_error(output: str, max_lines: int = 80):
    # Keep useful failure lines, not thousands of lm_eval WARNING lines.
    lines = output.splitlines()
    keys = ("ERROR", "Traceback", "ValueError", "RuntimeError", "CUDA out of memory",
            "No module", "ModuleNotFoundError", "ImportError", "OSError", "FAILED")
    important = [ln for ln in lines if any(k in ln for k in keys) and " WARNING " not in ln]
    selected = important[-max_lines:] if important else lines[-max_lines:]
    return "\n".join(selected)


def run(cmd, check=True, quiet=False, error_lines=80, **kw):
    # quiet=True captures stdout/stderr and only prints a compact error summary on failure.
    print(f"$ {cmd}")
    args = cmd if isinstance(cmd, list) else shlex.split(cmd)
    if quiet:
        proc = subprocess.run(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, **kw)
        if check and proc.returncode != 0:
            print("\n[command failed — compact error log]")
            print(_brief_error(proc.stdout, max_lines=error_lines))
            raise subprocess.CalledProcessError(proc.returncode, args, output=proc.stdout)
        return subprocess.CompletedProcess(args, proc.returncode, proc.stdout, "")

    proc = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, **kw)
    out_lines = []
    for line in proc.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
        out_lines.append(line)
    proc.wait()
    output = "".join(out_lines)
    if check and proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, args, output=output)
    return subprocess.CompletedProcess(args, proc.returncode, output, "")

# Mount Drive (no-op if already mounted or running outside Colab).
if USE_DRIVE and not Path(DRIVE_ROOT).is_dir():
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception as exc:
        print(f"WARN: no Drive ({exc}) — results will be ephemeral")

# Clone the official eval repo (idempotent).
if not Path(EVAL_DIR).is_dir():
    run(f"git clone https://github.com/babylm-org/babylm-eval {EVAL_DIR}")
else:
    run(f"git -C {EVAL_DIR} pull --ff-only")

# Symlink: babylm-eval/results/ -> Drive/.../eval_results/ (persistence).
if USE_DRIVE:
    drive_results = Path(RESULTS_DIR)
    drive_results.mkdir(parents=True, exist_ok=True)
    eval_results_dir = Path(EVAL_DIR) / "results"
    if eval_results_dir.is_symlink():
        eval_results_dir.unlink()
    elif eval_results_dir.exists():
        for item in eval_results_dir.iterdir():
            tgt = drive_results / item.name
            if not tgt.exists(): shutil.move(str(item), str(tgt))
        shutil.rmtree(eval_results_dir)
    eval_results_dir.symlink_to(drive_results, target_is_directory=True)
    print(f"symlink: {eval_results_dir} -> {drive_results}")

# Install lm-eval-harness + deps (~2 min on first run).
run("pip install -q --upgrade 'lm-eval[multilingual]>=0.4.4' transformers accelerate datasets sentencepiece protobuf")

# Sanity checks.
multi = Path(EVAL_DIR) / "multilingual"
print(f"\nmultilingual dir   : {multi}  exists={multi.is_dir()}")
print(f"tasks dir          : {(multi / 'tasks').is_dir()}  (custom YAML task defs)")
print(f"zeroshot script    : {(multi / 'scripts/zeroshot_model.sh').is_file()}")
print(f"finetune script    : {(multi / 'scripts/finetune_model.sh').is_file()}")
print(f"fast-all script    : {(multi / 'scripts/zeroshot_model_fast_all.sh').is_file()}")
print(f"collate script     : {(multi / 'scripts/collate_results.py').is_file()}")

# Probe that lm_eval is importable and the CLI works.
try:
    import lm_eval  # noqa: F401
    print(f"lm_eval version    : {lm_eval.__version__}")
except Exception as exc:
    print(f"[!] lm_eval import FAILED: {exc}")
    raise

# Probe that the custom tasks resolve (this is the most common silent failure).
print("\nProbing custom tasks (must list zeroshot_eng/nld/zho):")
os.chdir(multi)
probe = run("python3 -m lm_eval --tasks list --include_path tasks/", check=False, quiet=True)
matching_tasks = [line.strip() for line in probe.stdout.splitlines()
                  if "zeroshot_eng" in line or "zeroshot_nld" in line or "zeroshot_zho" in line]
for line in matching_tasks[:20]:
    print(f"  {line}")
if not any("zeroshot_eng" in line for line in matching_tasks):
    print("[!] Custom tasks NOT found. lm_eval cannot see multilingual/tasks/.")
    print("    Re-run this cell once. If it still fails, the BabyLM eval repo or lm_eval version changed.")
else:
    print("Environment ready.")

In [ ]:
# Cell 3 — HF token
# ------------------
# The eval pipeline downloads public HF models; a token is not strictly
# required, but it speeds up downloads (no rate-limit) and lets us authenticate.

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    if HF_TOKEN: print("HF_TOKEN loaded from Colab Secrets.")
except Exception:
    pass
if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    from getpass import getpass
    HF_TOKEN = getpass("HF_TOKEN (hidden): ").strip()

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN  # compat

from huggingface_hub import HfApi
me = HfApi(token=HF_TOKEN).whoami()
print(f"Authenticated as: {me.get('name', '<?>')}")

In [ ]:
# Cell 4 — Repair HF tokenizer metadata (run once before evaluation)
# ---------------------------------------------------------------
# Our uploaded repos already have tokenizer.model, but tokenizer_config.json
# incorrectly advertised it as PreTrainedTokenizerFast. Transformers then tries
# to parse the binary SentencePiece file as TikToken text and crashes.
# This cell uploads a correct slow SentencePiece/Llama tokenizer config.

REPAIR_TOKENIZER_METADATA = True
REPAIR_ALL_REVISIONS = True  # True also fixes chck_NM branches for H4 later.

if REPAIR_TOKENIZER_METADATA:
    import json, tempfile
    from pathlib import Path
    from huggingface_hub import HfApi
    from transformers import AutoTokenizer

    api = HfApi(token=HF_TOKEN)
    repair_dir = Path(tempfile.mkdtemp(prefix="hf_tokenizer_repair_"))

    tokenizer_config = {
        "tokenizer_class": "LlamaTokenizer",
        "bos_token": "<s>",
        "eos_token": "</s>",
        "pad_token": "<pad>",
        "unk_token": "<unk>",
        "model_max_length": 512,
        "add_bos_token": False,
        "add_eos_token": False,
        "legacy": True,
        "clean_up_tokenization_spaces": False,
    }
    special_tokens_map = {
        "bos_token": "<s>",
        "eos_token": "</s>",
        "pad_token": "<pad>",
        "unk_token": "<unk>",
    }
    generation_config = {
        "bos_token_id": 1,
        "eos_token_id": 2,
        "pad_token_id": 3,
        "_from_model_config": True,
    }

    files = {
        "tokenizer_config.json": tokenizer_config,
        "special_tokens_map.json": special_tokens_map,
        "generation_config.json": generation_config,
    }
    local_files = []
    for name, payload in files.items():
        p = repair_dir / name
        p.write_text(json.dumps(payload, indent=2), encoding="utf-8")
        local_files.append(p)

    repos_to_fix = sorted(set(FINAL_MODELS + H4_MODELS))
    for repo_id in repos_to_fix:
        print("=" * 72)
        print(f"Repairing tokenizer metadata: {repo_id}")
        refs = api.list_repo_refs(repo_id=repo_id, repo_type="model")
        revisions = [b.name for b in refs.branches]
        if not REPAIR_ALL_REVISIONS:
            revisions = ["main"]
        print(f"revisions: {len(revisions)}")

        for j, rev in enumerate(revisions, 1):
            for p in local_files:
                api.upload_file(
                    repo_id=repo_id,
                    repo_type="model",
                    revision=rev,
                    path_or_fileobj=str(p),
                    path_in_repo=p.name,
                    commit_message="Fix SentencePiece tokenizer metadata",
                )
            if j == 1 or j == len(revisions) or j % 5 == 0:
                print(f"  repaired {j}/{len(revisions)} revisions")

        # Smoke test main branch locally before running expensive eval.
        tok = AutoTokenizer.from_pretrained(repo_id, revision="main", use_fast=False, token=HF_TOKEN)
        ids = tok("Hello world", add_special_tokens=False).input_ids
        print(f"  smoke test ids: {ids[:10]} (n={len(ids)})")

    print("\nTokenizer metadata repair complete.")
else:
    print("REPAIR_TOKENIZER_METADATA=False — skipping.")

In [ ]:
# Cell 5 — Zero-shot eval on the FINAL checkpoint of each model
# -------------------------------------------------------------
# Tasks: BLiMP, MultiBLiMP, HellaSwag, Winogrande, XCOMPS, XStoryCloze, ZhoBLiMP.
# We call lm_eval directly instead of the shell wrapper so we can force
# use_fast_tokenizer=False and fail loudly if tokenizer/model loading breaks.

def find_result_files(model_name: str, revision: str = "main"):
    encoded = model_name.replace("/", "__")
    base = Path(EVAL_DIR) / "results" / revision
    candidates = [base / encoded, base / model_name]
    files = []
    for c in candidates:
        if c.is_dir():
            files.extend(c.glob("results_*.json"))
    if base.is_dir():
        files.extend([p for p in base.glob("results_*.json") if model_name.split("/")[-1] in p.name])
    return sorted(set(files))

def run_lm_eval(model: str, lang: str, revision: str = "main"):
    task = f"zeroshot_{lang}"
    model_args = (
        f"pretrained={model},revision={revision},trust_remote_code=True,"
        f"use_fast_tokenizer=False"
    )
    cmd = (
        f"python3 -m lm_eval --model hf --model_args {model_args} "
        f"--tasks {task} --device cuda --output_path ../results/{revision} "
        f"--batch_size auto:10 --num_fewshot 0 --log_samples --include_path tasks/"
    )
    # quiet=True suppresses massive lm_eval WARNING logs; failures still print compact errors.
    return run(cmd, check=True, quiet=True, error_lines=100)

if RUN_ZEROSHOT_FINAL:
    os.chdir(Path(EVAL_DIR) / "multilingual")
    print(f"cwd = {os.getcwd()}\n")

    failures = []
    langs = LANGS.split()
    for i, model in enumerate(FINAL_MODELS, 1):
        existing = find_result_files(model)
        if len(existing) >= len(langs):
            print(f"[{i}/{len(FINAL_MODELS)}] {model} — already evaluated ({len(existing)} files), skipping")
            continue

        print("=" * 72)
        print(f"[{i}/{len(FINAL_MODELS)}] zero-shot: {model}")
        print("=" * 72)

        before = set(find_result_files(model))
        ok = True
        for lang in langs:
            print(f"  [{lang}] running...", flush=True)
            try:
                run_lm_eval(model, lang, revision="main")
                print(f"  [{lang}] done")
            except subprocess.CalledProcessError as exc:
                print(f"  [{lang}] FAILED: exit code {exc.returncode}")
                ok = False
                break

        produced = set(find_result_files(model)) - before
        all_files = find_result_files(model)
        if not ok or len(all_files) < len(langs):
            print(f"\n[!] Incomplete eval for {model}: {len(all_files)} / {len(langs)} result files.")
            failures.append(model)
            continue
        print(f"OK: {model} ({len(all_files)} result files; {len(produced)} new)\n")

    print("=" * 72)
    print(f"DONE. Success: {len(FINAL_MODELS) - len(failures)} / {len(FINAL_MODELS)}")
    if failures:
        print("Failures:")
        for f in failures: print(f"  - {f}")
    print("=" * 72)
else:
    print("RUN_ZEROSHOT_FINAL=False — skipping this cell.")

In [ ]:
# Cell 6 — Zero-shot per intermediate checkpoint (H4 = acquisition order)
# ----------------------------------------------------------------------
# Iterates over chck_1M, chck_2M, ..., chck_600M for each H4 model. THIS IS SLOW.
# Uses the same direct lm_eval call as Cell 5 so tokenizer loading is stable.

if RUN_ZEROSHOT_INTERMEDIATE:
    from huggingface_hub import HfApi

    os.chdir(Path(EVAL_DIR) / "multilingual")
    api = HfApi(token=HF_TOKEN)
    langs = LANGS.split()
    failures = []

    for i, model in enumerate(H4_MODELS, 1):
        refs = api.list_repo_refs(repo_id=model, repo_type="model")
        revisions = [b.name for b in refs.branches if b.name.startswith("chck_")]
        revisions = sorted(revisions, key=lambda x: int(x.replace("chck_", "").replace("M", "")))

        print("=" * 72)
        print(f"[{i}/{len(H4_MODELS)}] intermediate: {model} ({len(revisions)} revisions)")
        print("=" * 72)

        for rev in revisions:
            existing = find_result_files(model, revision=rev)
            if len(existing) >= len(langs):
                print(f"{model} / {rev} — already evaluated ({len(existing)} files), skipping")
                continue

            ok = True
            for lang in langs:
                print(f"  [{rev}][{lang}] running...", flush=True)
                try:
                    run_lm_eval(model, lang, revision=rev)
                    print(f"  [{rev}][{lang}] done")
                except subprocess.CalledProcessError as exc:
                    print(f"  [{rev}][{lang}] FAILED: exit code {exc.returncode}")
                    ok = False
                    failures.append((model, rev, lang))
                    break
            if ok:
                print(f"OK: {model} / {rev}")

    print("=" * 72)
    print(f"DONE intermediate eval. Failures: {len(failures)}")
    for f in failures[:50]:
        print(f"  - {f[0]} / {f[1]} / {f[2]}")
    print("=" * 72)
else:
    print("RUN_ZEROSHOT_INTERMEDIATE=False — skipping (run this in another session).")

In [ ]:
# Cell 6 — Fine-tune eval (OPTIONAL, VERY SLOW)
# --------------------------------------------
# Each (model, language) pair takes ~1-3h. For 5 models x 3 languages = ~30h.
# The leaderboard accepts incomplete submissions (missing tasks = score 0).
# Recommendation: skip for now; run only on TAAM_v2 seed 42 at the end.

if RUN_FINETUNE:
    os.chdir(Path(EVAL_DIR) / "multilingual")
    for i, model in enumerate(FINAL_MODELS, 1):
        print("=" * 72)
        print(f"[{i}/{len(FINAL_MODELS)}] finetune: {model}")
        print("=" * 72)
        try:
            run(f"bash scripts/finetune_model.sh --model_name {model} --langs '{LANGS}'")
        except subprocess.CalledProcessError as exc:
            print(f"FAILED: {exc}")
            continue
        print(f"OK: {model}\n")
else:
    print("RUN_FINETUNE=False — skipping (run at the end, only on key models).")

In [ ]:
# Cell 7 — Build *_submission.json + *_predictions.json files
# -----------------------------------------------------------
# These are the artifacts uploaded to the leaderboard. --fast also includes
# results from intermediate checkpoints (only if Cell 5 was run).

import json

# These are the headline tasks the leaderboard expects per language.
# Subtask rows like "blimp_adjunct_island" are folded into "blimp" by lm_eval already.
EXPECTED_TASKS = {
    "eng": ["blimp", "hellaswag_en_mubench", "multiblimp_eng", "winogrande_en_mubench", "xstorycloze_en_mubench"],
    "nld": ["blimp_nl", "hellaswag_nl_mubench", "multiblimp_nld", "winogrande_nl_mubench", "xcomps_nl", "xstorycloze_nl_mubench"],
    "zho": ["hellaswag_zh_mubench", "winogrande_zh_mubench", "xcomps_zh", "xstorycloze_zh_mubench", "zhoblimp"],
}
ALL_EXPECTED = {t for tasks in EXPECTED_TASKS.values() for t in tasks}


def load_lm_eval_results(model: str, revision: str = "main") -> dict:
    """Merge all results_*.json files for one model into one dict {task: lm_eval_metrics}."""
    encoded = model.replace("/", "__")
    base = Path(EVAL_DIR) / "results" / revision
    folders = [base / encoded, base / model]
    merged: dict = {}
    for folder in folders:
        if not folder.is_dir():
            continue
        for jf in sorted(folder.glob("results_*.json")):
            try:
                data = json.loads(jf.read_text())
                merged.update(data.get("results", {}))
            except Exception as exc:
                print(f"  skipping {jf}: {exc}")
    return merged


# Map leaderboard-expected name -> list of lm_eval keys that should be tried (in order).
TASK_ALIASES = {
    "blimp": ["blimp", "blimp_babylm_filtered"],
}

def build_submission(raw_results: dict) -> tuple[dict, list[str]]:
    # IMPORTANT: keys must match the leaderboard's MULTILINGUAL_*_KEYS exactly.
    # The leaderboard's read_evals.py compares with `task_val.benchmark == k` literally,
    # so e.g. BLiMP EN must be stored under "blimp_babylm_filtered" (not the alias "blimp"),
    # otherwise the score is silently dropped to 0 in the leaderboard averages.
    submission = {}
    missing = []
    for lang, tasks in EXPECTED_TASKS.items():
        any_found = any(any(k in raw_results for k in TASK_ALIASES.get(t, [t])) for t in tasks)
        for t in tasks:
            candidates = TASK_ALIASES.get(t, [t])
            found_key = next((k for k in candidates if k in raw_results), None)
            if found_key:
                acc = (raw_results[found_key].get("acc,none")
                       or raw_results[found_key].get("acc")
                       or raw_results[found_key].get("accuracy"))
                if acc is not None:
                    submission[found_key] = {found_key: float(acc)}
                else:
                    missing.append(t)
            elif any_found:
                missing.append(t)
    return submission, missing


if RUN_COLLATE:
    submissions = Path(DRIVE_ROOT) / "submissions" if USE_DRIVE else Path("/content/submissions")
    submissions.mkdir(parents=True, exist_ok=True)

    for model in FINAL_MODELS:
        short = model.split("/")[-1]
        print("\n" + "=" * 72)
        print(f"collate: {model}")
        print("=" * 72)

        raw = load_lm_eval_results(model, revision="main")
        if not raw:
            print(f"  [!] no results loaded for {model}; skipping")
            continue

        submission, missing = build_submission(raw)
        if missing:
            print(f"  warning: {len(missing)} expected task(s) missing -> will be scored as 0:")
            for t in missing[:10]:
                print(f"    - {t}")

        # Submission file (scores only).
        out_sub = submissions / f"{short}_submission.json"
        out_sub.write_text(json.dumps(submission, indent=2))
        print(f"  -> submission:  {out_sub}  ({out_sub.stat().st_size} bytes, {len(submission)} tasks)")

        # Predictions file (raw zeroshot dict + placeholders for finetune/fast_eval).
        # IMPORTANT: always probe the disk for intermediate checkpoints regardless of
        # RUN_ZEROSHOT_INTERMEDIATE — the flag only controls whether to *run* the eval
        # (Cell 6), not whether to include already-evaluated checkpoints here.
        # Without fast_eval_results, the leaderboard tags the submission as
        # workshop-only instead of challenge entry.
        predictions = {"zeroshot": raw, "finetune": {}}
        fast = []
        revisions = (
            [f"chck_{i}M" for i in range(1, 10)]
            + [f"chck_{i*10}M" for i in range(1, 10)]
            + [f"chck_{i*100}M" for i in range(1, 11)]
        )
        for rev in revisions:
            rev_raw = load_lm_eval_results(model, revision=rev)
            if rev_raw:
                rev_sub, _ = build_submission(rev_raw)
                if rev_sub:
                    fast.append({k: v[k] for k, v in rev_sub.items()})
        if fast:
            predictions["fast_eval_results"] = fast
            print(f"  -> fast_eval_results: {len(fast)} revisions found on disk")
        else:
            print(f"  -> no fast_eval_results on disk for this model (workshop-only)")

        out_pred = submissions / f"{short}_predictions.json"
        out_pred.write_text(json.dumps(predictions, indent=2))
        print(f"  -> predictions: {out_pred}  ({out_pred.stat().st_size} bytes)")

    print("\n" + "=" * 72)
    print(f"Files in {submissions}:")
    print("=" * 72)
    for p in sorted(submissions.iterdir()):
        print(f"  {p.name:60s}  {p.stat().st_size:>10} bytes")
else:
    print("RUN_COLLATE=False — skipping.")

In [ ]:
# Cell 8 — Comparative table of ALL evaluated models (auto-discover)
# ------------------------------------------------------------------
# Scans results/main/ for every model that has been evaluated, regardless of
# what FINAL_MODELS contains right now. This way running with FINAL_MODELS
# restricted to a single new model still produces a complete comparison and
# never overwrites prior models' columns.

if RUN_SUMMARY_TABLE:
    import json
    import pandas as pd

    # Auto-discover every evaluated model from results/main/.
    main_dir = Path(EVAL_DIR) / "results" / "main"
    discovered = []
    if main_dir.is_dir():
        for folder in sorted(main_dir.iterdir()):
            if not folder.is_dir():
                continue
            # lm_eval folders look like "org__short" (we used "__" as separator).
            if "__" in folder.name:
                org, short = folder.name.split("__", 1)
                discovered.append(f"{org}/{short}")
            else:
                discovered.append(folder.name)

    if not discovered:
        print("No evaluated models found in results/main/.")
    else:
        print(f"Found {len(discovered)} evaluated model(s):")
        for m in discovered:
            print(f"  - {m}")

    rows = []
    for model in discovered:
        result_files = find_result_files(model, revision="main")
        if not result_files:
            print(f"WARN: no result files for {model}")
            continue
        for jf in sorted(set(result_files)):
            try:
                data = json.loads(jf.read_text())
            except Exception as exc:
                print(f"  skipping {jf}: {exc}")
                continue
            results = data.get("results", {})
            for task, metrics in results.items():
                acc = metrics.get("acc,none") or metrics.get("acc") or metrics.get("accuracy")
                if acc is None: continue
                rows.append({
                    "model": model.split("/")[-1].replace("babylm-2026-", ""),
                    "task":  task,
                    "acc":   round(float(acc) * 100, 2),
                })

    if not rows:
        print("No results yet — run Cell 5 first.")
    else:
        df = pd.DataFrame(rows)
        pivot = df.pivot_table(index="task", columns="model", values="acc", aggfunc="first")
        # Preferred column order; any unknown columns are appended.
        preferred = ["b0-seed42", "b0-seed1337",
                     "taam-seed42", "taam-seed1337",
                     "taam_v2-seed42", "taam_v2-seed1337"]
        cols = [c for c in preferred if c in pivot.columns]
        cols += [c for c in pivot.columns if c not in cols]
        pivot = pivot[cols]
        print("\n" + "=" * 88)
        print(f"COMPARISON — accuracy (%) by task/model (main checkpoint, {len(cols)} models)")
        print("=" * 88)
        print(pivot.to_string())
        if USE_DRIVE:
            out = Path(DRIVE_ROOT) / "submissions" / "comparison_final.csv"
            out.parent.mkdir(parents=True, exist_ok=True)
            pivot.to_csv(out)
            print(f"\nSaved to: {out}  ({len(cols)} columns)")
else:
    print("RUN_SUMMARY_TABLE=False — skipping.")